# 12. MLflow - Tracking 심화 (wine)

PDF `MLflow.pdf` 29~43쪽 실습

11번에서는 breast_cancer로 "기록한다"를 했고, 여기서는 wine 데이터로 **무엇을 어떻게 기록할지**를 나눠서 다룬다.

| 쪽 | 내용 |
|---|---|
| 29 | MlflowClient로 서버 조회 |
| 31~33 | 데이터 준비, Pipeline 만들기 |
| 34~36 | 파라미터 / 지표 / 태그 |
| 38~41 | Artifact (JSON, 텍스트, 그림, 데이터셋, 환경) |
| 42 | step이 있는 지표 |
| 43 | 실패한 Run |

**준비**: 11번에서 띄운 Tracking 서버가 계속 떠 있어야 한다. 꺼졌으면 터미널에서 다시 실행한다.

```powershell
cd C:\Users\samsung1\git_project\samsung_practice\머신러닝분류\mlflow_practice
C:\Users\samsung1\git_project\samsung_practice\.venv\Scripts\mlflow.exe server --host 127.0.0.1 --port 5000 --backend-store-uri sqlite:///mlflow.db --serve-artifacts --artifacts-destination ./artifacts
```

커널은 `samsung_practice (.venv)`.

## 1. 서버 연결 확인 (29쪽)

`MlflowClient`는 `mlflow.log_*` 보다 한 단계 아래에 있는 **조회/관리용 객체**다.
지금은 "서버에 잘 붙었나" 확인용으로만 쓴다.

교재는 주소를 환경변수 `MLFLOW_TRACKING_URI`로 관리한다. 나중에 82쪽 `mlflow run`이나
93쪽 CLI 실습에서 같은 환경변수를 쓰므로 여기서 습관을 들여 둔다.

In [5]:
import os
import mlflow
from mlflow import MlflowClient

# 교재는 환경변수로 주소를 관리한다. 노트북에서는 직접 넣어 준다.
os.environ["MLFLOW_TRACKING_URI"] = "http://127.0.0.1:5000"

uri = os.environ["MLFLOW_TRACKING_URI"]
mlflow.set_tracking_uri(uri)

client = MlflowClient(tracking_uri=uri)
experiment = client.get_experiment_by_name("wine-classification")

print("Tracking URI:", uri)
if experiment is not None:
    print("Artifacts:", experiment.artifact_location)
else:
    print("아직 wine-classification 실험이 없다. 아래에서 만든다.")

Tracking URI: http://127.0.0.1:5000
Artifacts: mlflow-artifacts:/5


## 2. 데이터 준비 (31~32쪽)

> **교재에서 한 곳 고친다.**
> 32쪽은 `X = wine.data`로 되어 있는데 이러면 `X`가 numpy 배열이 된다.
> 그런데 40·41쪽에서 `X_train.assign(...)`, `X_train.index.tolist()`처럼
> **DataFrame에만 있는 기능**을 쓴다. 그대로 두면 40쪽에서
> `AttributeError: 'numpy.ndarray' object has no attribute 'assign'`이 난다.
> 그래서 `load_wine(as_frame=True)`로 불러와 DataFrame으로 받는다.

`SEED`도 교재 33쪽부터 쓰이는데 정의가 없어서 여기서 먼저 잡아 둔다.

In [6]:
import numpy as np
import pandas as pd
import mlflow.sklearn

from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

SEED = 42

In [7]:
wine = load_wine(as_frame=True)   # 교재 32쪽의 load_wine() 에서 고친 부분
X = wine.data
y = wine.target

# 먼저 test를 떼고, 남은 train에서 다시 val을 뗀다 (3등분)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED
)
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=0.2, stratify=y_train, random_state=SEED
)

print(len(X_tr), len(X_val), len(X_test))
print(type(X_train).__name__, X_train.shape)   # DataFrame 이어야 한다

113 29 36
DataFrame (142, 13)


## 3. 모델 만드는 함수 (33쪽)

스케일링과 모델을 `Pipeline`으로 묶는다. `C`만 바꿔가며 여러 번 만들 것이므로 함수로 둔다.

`C`는 LogisticRegression의 규제 강도로, **작을수록 규제가 세다**. 64쪽 실습에서 이 값을 훑는다.

In [8]:
def make_model(C=1.0):
    return Pipeline([
        ("scale", StandardScaler()),
        ("model", LogisticRegression(
            C=C, max_iter=2000, random_state=SEED))
    ])


pipe = make_model()
pipe.fit(X_tr, y_tr)
pred = pipe.predict(X_val)

print("val f1_macro:", f1_score(y_val, pred, average="macro"))

val f1_macro: 1.0


## 4. 기준 모델 Run (34쪽)

이 Run의 `run_id`를 **`baseline_id`에 저장해 둔다.** 38~41쪽에서 이 Run에 내용을 계속 덧붙일 것이다.

> ### 교재와 다르게 간다 - 학습 데이터를 X_tr로 바꿨다
>
> 교재 34쪽은 `pipe.fit(X_train, y_train)`으로 학습하고 `X_val`로 평가한다.
> 그런데 `X_val`은 32쪽에서 `X_train`을 쪼개서 만든 조각이라,
> **평가용 29행이 학습 데이터에 그대로 들어가 있다.** (겹침 29/29)
>
> **교재 실태**: `X_tr`은 32쪽에서 만들고 33쪽에서 딱 한 번 쓴 뒤 끝까지 안 나온다.
> `fit(X_train, y_train)`이 34·36·39·46·47·64·74·75·78쪽으로 교재의 기본 패턴이다.
> 즉 교재는 3등분을 해 놓고 `X_tr`을 사실상 버린 채 `X_val`만 평가용으로 계속 쓴다.
>
> **그래도 고치는 이유는 64~68쪽 때문이다.** 거기서 `C`를 4개 훑은 뒤
> **`val_f1_macro`가 가장 높은 Run을 골라** 최종 모델로 등록한다.
> 고르는 기준이 되는 점수가 학습에 쓴 데이터에서 나온 것이면 그 선택 자체가 편향된다.
> 단순히 기록만 하는 거라면 넘어가도 되지만, **선택에 쓰는 점수**라 문제가 된다.
>
> 그래서 이 노트북은 33쪽 방식(`X_tr` 학습)으로 통일한다.
> 고친 셀은 34·36·39·40·41쪽이고, 각 셀 주석에 교재 원본이 무엇이었는지 적어 뒀다.
> **교재와 달라지는 건 `fit()`에 넣는 인자 하나뿐이다.**

> ### 주의 - 여기서는 점수가 안 변한다
>
> 고치기 전과 후의 점수를 실제로 비교하면 이렇다.
>
> | 학습 데이터 | val f1 | test f1 |
> |---|---|---|
> | `X_tr` (고친 것) | 1.0 | 0.971 |
> | `X_train` (교재) | 1.0 | 0.971 |
>
> **차이가 없다.** wine은 13개 특성으로 3종류를 가르는 쉬운 데이터라
> 어느 쪽으로 학습해도 거의 다 맞힌다. (5-fold CV로 봐도 0.97~1.0)
>
> 즉 f1이 1.0인 건 데이터가 쉬워서이지 데이터가 새서가 아니다.
> **증상이 안 나타난다고 방법이 맞는 건 아니다.** 어려운 데이터였다면
> 교재 방식은 실제보다 높은 점수를 보여줬을 것이고, 그걸 믿고 모델을 고르면 틀린 선택을 하게 된다.

정리하면 이 노트북에서는 이렇게 쓴다.

| 이름 | 행 수 | 용도 |
|---|---|---|
| `X_tr` | 113 | **학습** |
| `X_val` | 29 | 검증 (성능 비교, 모델 선택) |
| `X_test` | 36 | 최종 평가 (마지막까지 안 건드림) |

In [ ]:
mlflow.set_tracking_uri("http://127.0.0.1:5000")
exp = mlflow.set_experiment("wine-classification")

with mlflow.start_run(run_name="lr-baseline") as run:
    pipe = make_model(C=1.0)
    # 교재는 fit(X_train, y_train). X_val이 X_train에 포함돼 있어 평가가 무의미해진다.
    # 33쪽과 같이 X_tr로 학습해야 맞다.
    pipe.fit(X_tr, y_tr)
    pred = pipe.predict(X_val)

    mlflow.log_param("C", 1.0)
    mlflow.log_metric("val_f1_macro", f1_score(y_val, pred, average="macro"))

    baseline_id = run.info.run_id

print("baseline_id =", baseline_id)
print("val f1_macro:", f1_score(y_val, pred, average="macro"))

🏃 View run lr-baseline at: http://127.0.0.1:5000/#/experiments/5/runs/88a3d9b42f3a4b1489bfbb5b728265e1
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/5
baseline_id = 88a3d9b42f3a4b1489bfbb5b728265e1
val f1_macro: 1.0


## 5. 파라미터 (35쪽)

`log_params`로 한 번에 여러 개를 남긴다. 나중에 "이 Run이 어떤 설정이었나"를 되짚는 근거가 된다.

핵심은 **파라미터는 Run 안에서 한 번 정해지면 못 바꾼다**는 것. 같은 키에 다른 값을 넣으면 에러가 난다.
설정을 바꾸려면 값을 덮어쓰는 게 아니라 **새 Run을 시작해야 한다.**

In [10]:
with mlflow.start_run(run_name="params-demo"):
    mlflow.log_params({
        "model_type": "logistic_regression",
        "C": 1.0, "seed": SEED,
        "split_id": "wine-seed42-v1"
    })

    # 같은 키에 다른 값 -> 에러. 직접 확인해 본다.
    try:
        mlflow.log_param("C", 10.0)
        print("덮어쓰기 성공 (이러면 안 되는데?)")
    except Exception as error:
        print(type(error).__name__)
        print(str(error)[:120])

MlflowException
INVALID_PARAMETER_VALUE: Changing param values is not allowed. Param with key='C' was already logged with value='1.0' fo
🏃 View run params-demo at: http://127.0.0.1:5000/#/experiments/5/runs/33bed33af702413f9d04ed24fbd7421e
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/5


## 6. 지표와 태그 (36쪽)

- `log_metrics` : 숫자. 정렬·비교·그래프가 되는 값
- `set_tags` : 문자열 메모. 검색으로 걸러낼 때 쓴다

`purpose` 같은 태그를 붙여 두면 67쪽에서 `tags.purpose = '...'` 로 원하는 Run만 뽑을 수 있다.

In [11]:
with mlflow.start_run(run_name="metrics-tags"):
    pipe = make_model().fit(X_tr, y_tr)   # 교재는 X_train. 34쪽과 같은 이유로 고침
    pred = pipe.predict(X_val)

    mlflow.log_metrics({
        "val_accuracy": accuracy_score(y_val, pred),
        "val_f1_macro": f1_score(y_val, pred, average="macro")
    })

    mlflow.set_tags({"purpose": "baseline", "owner": "student"})
    mlflow.set_tag("decision", "비교 기준 모델")

print("기록 완료 | val f1_macro:", f1_score(y_val, pred, average="macro"))

🏃 View run metrics-tags at: http://127.0.0.1:5000/#/experiments/5/runs/3424fc2b62b740c1a7d19fea3170ee1d
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/5
기록 완료 | val f1_macro: 1.0


## 7. Artifact - 파일로 남기기 (38쪽)

여기부터는 **이미 끝난 Run을 다시 열어서** 내용을 덧붙인다.
`start_run(run_id=baseline_id)` 처럼 `run_id`를 주면 새 Run이 아니라 그 Run이 이어진다.

- `log_dict` : 딕셔너리를 JSON 파일로
- `log_text` : 문자열을 텍스트 파일로

In [12]:
with mlflow.start_run(run_id=baseline_id):
    mlflow.log_dict({
        "dataset": "sklearn-wine",
        "split_id": "wine-seed42-v1",
        "primary_metric": "val_f1_macro"
    }, "config/evaluation.json")

    mlflow.log_text(
        "StandardScaler와 LogisticRegression 기준 모델",
        "notes/baseline.txt")

print("38쪽 완료 - UI에서 baseline Run의 Artifacts 탭 확인")

🏃 View run lr-baseline at: http://127.0.0.1:5000/#/experiments/5/runs/88a3d9b42f3a4b1489bfbb5b728265e1
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/5
38쪽 완료 - UI에서 baseline Run의 Artifacts 탭 확인


### 그림 남기기 (39쪽)

`log_figure`는 matplotlib figure를 바로 이미지로 저장한다. 파일로 먼저 저장할 필요가 없다.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

# 교재는 X_train으로 다시 학습한다. 그러면 baseline Run에 붙는 그림이
# 정작 baseline 모델의 그림이 아니게 된다. 같은 X_tr로 학습해야 앞뒤가 맞는다.
pipe = make_model().fit(X_tr, y_tr)
pred = pipe.predict(X_val)

with mlflow.start_run(run_id=baseline_id):
    fig, ax = plt.subplots(figsize=(5, 4))
    ConfusionMatrixDisplay.from_predictions(y_val, pred, ax=ax)
    mlflow.log_figure(fig, "plots/confusion_matrix.png")
    plt.close(fig)

print("39쪽 완료")

🏃 View run lr-baseline at: http://127.0.0.1:5000/#/experiments/5/runs/88a3d9b42f3a4b1489bfbb5b728265e1
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/5
39쪽 완료


### 어떤 데이터로 학습했는지 남기기 (40쪽)

`mlflow.data.from_pandas`로 데이터셋 자체를 등록하면 UI의 **Datasets** 항목에 잡힌다.
"이 모델이 어떤 데이터로 학습됐는지"를 Run에 묶어 두는 것이다.

In [ ]:
# 교재는 X_train을 기록한다. 실제로 학습에 쓴 건 X_tr이므로 그쪽을 남긴다.
train_df = X_tr.assign(target=y_tr)
train_ds = mlflow.data.from_pandas(train_df, name="wine-train", targets="target")

with mlflow.start_run(run_id=baseline_id):
    mlflow.log_input(train_ds, context="training")
    mlflow.set_tag("data_version", "sklearn-wine-v1")

print("40쪽 완료 |", train_ds.name, train_df.shape)

🏃 View run lr-baseline at: http://127.0.0.1:5000/#/experiments/5/runs/88a3d9b42f3a4b1489bfbb5b728265e1
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/5
40쪽 완료 | wine-train (113, 14)


c:\Users\samsung1\git_project\samsung_practice\.venv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


### 재현에 필요한 나머지 (41쪽)

나중에 같은 결과를 다시 만들려면 코드만으로는 부족하다.

- `pip freeze` : 어떤 라이브러리 버전이었는지
- split 인덱스 : 정확히 어떤 행이 train/val/test 였는지

둘 다 파일로 붙여 둔다.

In [ ]:
import subprocess
import sys

versions = subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True)

with mlflow.start_run(run_id=baseline_id):
    mlflow.log_text(versions, "env/requirements.txt")
    mlflow.log_dict({
        # 교재는 train에 X_train을 넣는데, 그러면 train 목록 안에 val 29행이
        # 그대로 다시 들어가 세 목록이 겹친다. 학습에 쓴 X_tr을 넣어야 맞다.
        "train": X_tr.index.tolist(),
        "val": X_val.index.tolist(),
        "test": X_test.index.tolist()
    }, "data/split_indices.json")

print("41쪽 완료 | pip freeze", len(versions.splitlines()), "줄")
print("겹치는 행 수:", len(set(X_tr.index) & set(X_val.index)), "(0이어야 정상)")

🏃 View run lr-baseline at: http://127.0.0.1:5000/#/experiments/5/runs/88a3d9b42f3a4b1489bfbb5b728265e1
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/5
41쪽 완료 | pip freeze 134 줄
겹치는 행 수: 0 (0이어야 정상)


## 8. step이 있는 지표 (42쪽)

`log_metric`에 `step`을 주면 **같은 이름의 지표를 여러 번** 남길 수 있다.
에폭별 loss처럼 시간에 따라 변하는 값에 쓴다. UI에서는 꺾은선 그래프로 보인다.

`client.get_metric_history`로 그 이력을 코드에서 다시 꺼낼 수 있다.

In [16]:
from mlflow import MlflowClient

with mlflow.start_run(run_name="step-demo") as run:
    for step, score in enumerate([0.70, 0.78, 0.76]):
        mlflow.log_metric("demo_score", score, step=step)
    history_id = run.info.run_id

client = MlflowClient()
history = client.get_metric_history(history_id, "demo_score")
print([(m.step, m.value) for m in history])

🏃 View run step-demo at: http://127.0.0.1:5000/#/experiments/5/runs/55b9d18c20484219a01b8b4174df252a
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/5
[(0, 0.7), (1, 0.78), (2, 0.76)]


## 9. 실패한 Run (43쪽)

학습 도중 에러가 나면 그 Run은 어떻게 될까. 일부러 에러를 내서 확인한다.

`with` 블록 안에서 예외가 나면 MLflow가 Run을 **`FAILED` 상태로 닫는다.**
열린 채로 남지 않는다는 게 중요하다 (`active_run()`이 `None`).

실패한 Run도 기록은 남으므로, 67쪽에서 Run을 고를 때
`attributes.status = 'FINISHED'` 조건으로 걸러내야 하는 이유가 여기 있다.

In [ ]:
try:
    with mlflow.start_run(run_name="failure-demo") as failed_run:
        failed_id = failed_run.info.run_id
        mlflow.set_tag("purpose", "error-demo")
        raise ValueError("교육용 의도적 오류")
except ValueError as error:
    print("잡힌 예외:", error)

print("active_run:", mlflow.active_run())          # None
print("서버 기록 status:", client.get_run(failed_id).info.status)   # FAILED

## 10. 정리 - UI에서 확인 (29~43쪽 마무리)

http://127.0.0.1:5000 에서 `wine-classification` 실험을 연다.

**확인할 것**

1. Run 5개 — `lr-baseline`, `params-demo`, `metrics-tags`, `step-demo`, `failure-demo`
2. `failure-demo`의 상태가 **FAILED**로 빨갛게 표시되는지
3. `lr-baseline`을 클릭 → **Artifacts** 탭에 5개 폴더가 쌓였는지
   - `config/evaluation.json`, `notes/baseline.txt`, `plots/confusion_matrix.png`,
     `env/requirements.txt`, `data/split_indices.json`
   - Run 하나에 여러 셀이 나눠서 붙인 결과다
4. `lr-baseline`의 **Datasets** 항목에 `wine-train`이 잡혔는지
5. `step-demo` → `demo_score`를 누르면 점 3개짜리 **꺾은선 그래프**가 나오는지
   (다른 지표는 점 하나짜리인 것과 비교)

**이 구간의 요점**

| 남기는 것 | 함수 | 성질 |
|---|---|---|
| 설정값 | `log_param(s)` | 한 번 정하면 못 바꿈 |
| 성능 숫자 | `log_metric(s)` | step으로 여러 번 가능 |
| 검색용 메모 | `set_tag(s)` | 문자열, 나중에 수정 가능 |
| 파일 | `log_dict` / `log_text` / `log_figure` | Run에 계속 덧붙일 수 있음 |
| 데이터 출처 | `log_input` | 어떤 데이터로 학습했는지 |

다음은 **44쪽부터 - 모델 자체를 저장하고 다시 불러오기**.
지금까지는 "기록"만 했고, 이제 학습된 모델을 남겨서 재사용하는 단계로 넘어간다.